# Lab 1b · RAG on a real public k8s dataset (free Kaggle T4)

**Layer 3 · Data — bonus.** Lab 1 used a tiny *synthetic* runbook so the
hallucinate-vs-grounded contrast was crisp. This lab runs the **same pipeline** on
real, messy, public data at scale:

> **[`mcipriano/stackoverflow-kubernetes-questions`](https://huggingface.co/datasets/mcipriano/stackoverflow-kubernetes-questions)**
> — 30,044 real Kubernetes Q&A pairs scraped from Stack Overflow (top-voted answer
> per question). License **CC-BY-SA-4.0** (attribution + share-alike).

What changes with real data:
- **Cleaning matters** — answers are raw HTML; we strip it (Phase 2).
- **Retrieval can miss** — at thousands of docs, recall@k drops below 1.0, unlike
  the 6-doc toy set. This is where eval gets real.
- **Index choice matters** — we use **HNSW** instead of Flat (Phase 4).

**Settings:** Accelerator = **GPU (T4)**, Internet = **On**. Then **Run All**.

In [ ]:
!pip install -q faiss-cpu sentence-transformers datasets beautifulsoup4

import torch, faiss, re, html, textwrap
import numpy as np
from bs4 import BeautifulSoup
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('cuda', torch.cuda.is_available())

## 1 · Load the dataset (Phase 1 · Sources)

We pull the dataset from the Hugging Face Hub and take a working **subset** (the
full 30k embeds fine on a T4, but a subset keeps the lab to a few minutes — bump
`N` if you want the whole thing). We inspect the columns first; real datasets rarely
match your assumptions.

In [ ]:
from datasets import load_dataset
ds = load_dataset('mcipriano/stackoverflow-kubernetes-questions', split='train')
print('rows:', len(ds))
print('columns:', ds.column_names)
print('\nfirst row (truncated):')
row = ds[0]
for k, v in row.items():
    print(f'  {k}: {str(v)[:120]}')

N = 3000
ds = ds.select(range(min(N, len(ds))))
print(f'\nusing {len(ds)} rows')

## 2 · Clean the HTML (Phase 2)

Stack Overflow answers are HTML — `<p>`, `<code>`, `<a>` tags, entities. Garbage in,
garbage out: we strip tags to plain text before embedding, and drop empties. This
is the *clean* half of Phase 2 that the toy corpus never needed.

In [ ]:
def clean(t):
    if not t:
        return ''
    t = BeautifulSoup(html.unescape(str(t)), 'html.parser').get_text(' ')
    return re.sub(r'\s+', ' ', t).strip()

# columns are Question / Answer (+ authors); be defensive about names
qcol = 'Question' if 'Question' in ds.column_names else ds.column_names[0]
acol = 'Answer' if 'Answer' in ds.column_names else ds.column_names[-1]

docs = []
for i, r in enumerate(ds):
    q, a = clean(r.get(qcol)), clean(r.get(acol))
    if len(a) < 40:        # skip near-empty answers
        continue
    docs.append({'doc_id': f'so-{i}', 'question': q, 'text': a})
print(f'{len(docs)} usable Q&A docs after cleaning')
print('\nexample cleaned answer:\n', textwrap.fill(docs[0]['text'][:300], 90))

## 3 · Chunk + embed (Phases 2–3)

We index each **answer** as the retrievable unit (and keep its question as
metadata). Long answers are split with the LLM tokenizer; short ones stay whole.
Same `bge-small` embedder and query prefix as Lab 1 — the rules don't change with
scale.

In [ ]:
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
EMB_ID = 'BAAI/bge-small-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
tok = AutoTokenizer.from_pretrained(MODEL_ID)
embedder = SentenceTransformer(EMB_ID, device=DEVICE)

def chunk_text(text, size=200, overlap=40):
    ids = tok.encode(text, add_special_tokens=False); step = size - overlap; out = []
    for s in range(0, len(ids), step):
        piece = ids[s:s+size]
        if not piece: break
        out.append(tok.decode(piece))
        if s + size >= len(ids): break
    return out

chunks = []
for d in docs:
    for j, c in enumerate(chunk_text(d['text'])):
        chunks.append({'chunk_id': f"{d['doc_id']}::{j}", 'doc_id': d['doc_id'],
                       'question': d['question'], 'text': c})
print(f'{len(chunks)} chunks from {len(docs)} answers')

vecs = embedder.encode([c['text'] for c in chunks], normalize_embeddings=True,
                       convert_to_numpy=True, batch_size=128, show_progress_bar=True).astype('float32')
print('embedded', vecs.shape)

## 4 · Vector store — **HNSW** this time (Phase 4)

At thousands of chunks a flat scan still works but starts to cost; **HNSW** (a
navigable graph) gives near-exact recall at a fraction of the query time — the
default for real corpora. The retrieval code is identical to Lab 1's Flat index;
only the index type changed.

In [ ]:
dim = vecs.shape[1]
index = faiss.IndexHNSWFlat(dim, 32)         # 32 = graph neighbours (M)
index.hnsw.efConstruction = 200
index.metric_type = faiss.METRIC_INNER_PRODUCT
index.add(vecs)
index.hnsw.efSearch = 64                       # higher = better recall, slower
print('HNSW index holds', index.ntotal, 'vectors')

def retrieve(query, k=4):
    qv = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True,
                         convert_to_numpy=True).astype('float32')
    sc, ii = index.search(qv, k)
    return [{**chunks[int(i)], 'score': float(s)} for s, i in zip(sc[0], ii[0]) if i != -1]

for h in retrieve('How do I fix a pod stuck in CrashLoopBackOff?', k=4):
    print(f"{h['score']:.3f}  {h['chunk_id']:<12} {h['text'][:70]}...")

## 5 · The LLM + RAG (Phases 5–6)

Same grounded-answer setup as Lab 1: retrieve real SO answers, stitch them in as
context, and let Qwen synthesize one cited answer. Now the model is grounded in
*community knowledge*, not a hand-written runbook.

In [ ]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
model.eval()

def generate(messages, max_new_tokens=220):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()

SYSTEM_RAG = ('You are a Kubernetes assistant. Answer using ONLY the Stack Overflow context below. '
              'Cite the chunk ids you used in square brackets. '
              "If the answer is not in the context, say you don't know.")

def ask(question, k=4):
    hits = retrieve(question, k=k)
    ctx = '\n\n'.join(f"[{h['chunk_id']}] {h['text']}" for h in hits)
    ans = generate([{'role': 'system', 'content': SYSTEM_RAG},
                    {'role': 'user', 'content': f'CONTEXT:\n{ctx}\n\nQUESTION: {question}'}])
    return ans, hits

ans, hits = ask('Why would a pod be stuck in Pending state and how do I debug it?', k=4)
print('SOURCES:', [h['chunk_id'] for h in hits])
print('\nANSWER:\n', textwrap.fill(ans, 90))

## 6 · Real-data reality check

Try a few questions. Watch for the failure modes the toy corpus hid:
- **noisy retrieval** — a chunk may be loosely related, not on-point;
- **conflicting answers** — Stack Overflow has many opinions; RAG surfaces several;
- **stale advice** — answers may predate current k8s APIs (the **freshness** problem
  from Phase 8).

In [ ]:
for q in ['How do I expose a deployment as a service?',
          'What does ImagePullBackOff mean and how do I fix it?',
          'How can I see the logs of a crashed container?']:
    ans, hits = ask(q, k=4)
    print('Q:', q)
    print('   sources:', [h['chunk_id'] for h in hits])
    print('   ', textwrap.fill(ans, 86).replace('\n', '\n    '), '\n')

## Takeaways

- **Same pipeline, real data.** The chunk→embed→index→retrieve→generate code is
  identical to Lab 1 — only the source (30k real SO Q&A) and the index (**HNSW**)
  changed.
- **Cleaning is no longer optional** — raw HTML had to be stripped before
  embedding (Phase 2).
- **Retrieval gets hard at scale** — relevant-but-imperfect chunks, conflicting
  answers, and **stale** advice all appear. This is why Lab 2's eval (recall@k,
  faithfulness) matters more here than on the toy set.
- **Freshness + provenance** (Phase 8) are real: cite sources, and re-index as the
  corpus ages.

*Data: [mcipriano/stackoverflow-kubernetes-questions](https://huggingface.co/datasets/mcipriano/stackoverflow-kubernetes-questions),
derived from Stack Overflow, CC-BY-SA-4.0.*